# 00 — Data audit

Field eligibility, availability drift, and the leakage alarm. Run this before anything else: it decides which columns are allowed to become features.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from src.config import load_config

cfg = load_config()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

## 1. Load a slice and check the schema

151 columns. Read a sample first — the full file is 2.26M rows.

In [ ]:
from src.data_load import load_accepted, profile_columns

acc = load_accepted(cfg['paths']['raw_accepted'], nrows=50_000)
print(f'{len(acc.columns)} columns')
profile_columns(acc, by_quarter=False).head(20)

## 2. Field eligibility — three categories, not two

Category B is the subtle one: `grade`/`sub_grade`/`int_rate` are known at origination so they are not leakage, but they are LendingClub's *model output*. Using them means learning to imitate the platform's scorecard. They are reserved as the benchmark instead.

In [ ]:
from src.leakage import eligible_features

buckets = eligible_features(list(acc.columns))
for k, v in buckets.items():
    print(f'{k}: {len(v)}')
print()
print('platform decision fields (benchmark only):', buckets['platform_decision'])

## 3. Availability drift

A feature whose *availability* tracks the issue date looks predictive out-of-time for a reason that has nothing to do with credit risk. Several bureau aggregates are near-empty before ~2015 and populated after.

In [ ]:
from src.leakage import availability_drift
from src.labeling import parse_issue_date

acc['issue_dt'] = parse_issue_date(acc['issue_d'])
availability_drift(acc, buckets['eligible'][:40], freq='Y').head(15)

## 4. Leakage alarm

IV above 0.50 against a 36-month default target is a bug report, not a discovery. Verified result: 9 fields flagged, all 9 post-origination. `last_fico_range_high` scores **3.15** — it is the borrower's *most recent* score, which collapses after a charge-off, and it sits one underscore from the legitimate `fico_range_high`.

In [ ]:
from src.binning import profile_features
from src.leakage import leakage_report

# Deliberately include post-origination fields to prove the alarm fires.
# In 01 onward they are excluded.
probe = [c for c in buckets['eligible'][:70] + buckets['post_origination']
         if acc[c].notna().mean() > 0.3]
prof = profile_features(acc.dropna(subset=['loan_status']).assign(
    target=(acc['loan_status'] == 'Charged Off').astype(int)), probe, 'target', cfg)
leakage_report(prof[prof['iv'].notna()], cfg)

## 5. Reconcile `docs/data_dictionary.md`

Anything whose fill rate jumps mid-panel belongs in the availability-drift section of `docs/leakage_fields.md`, not in the feature set.